In [13]:
import os
import sys
from pathlib import Path


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가한다.
def find_project_root() -> Path:
    """현재 작업 디렉터리에서 프로젝트 루트를 탐색한다."""
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr


PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "true")


'true'

In [14]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

dummy_user_index_md = """# 멤버 1번 일일 소비 분석 종합 리포트

- 분석 기준일: 2024-04-01
- 비교 기준일: 2024-03-31

## [1] 클리핑 데이터 → 안정적 지표 분석
- 과거 일평균(안정): 51,014원
- 오늘 총 지출액: 133,044원
- 평소 대비 지출 증가율: +160.80%

### 카테고리 비중 변화
#### 비중 증가 상위
- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)
- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)
- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)

#### 비중 감소 상위
- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)
- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)

## [2] 원본 데이터 → 행동 및 이상 탐지
- 과거 원본 일평균(전체): 104,424원
- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)

### 특이 지출 내역
- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활

## [3] 전날(2024-03-31) 대비 소비 비교 분석
- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)
- 결제 건수 비교: 6건 -> 9건 (+3건)
- 주 소비 카테고리 변화: 식비 -> 생활

## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
- 오늘의 소비 피크 타임: 2.오전(06-11)

### 시간대별 세부 비교
- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)
- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)
- 4.저녁(17-21): 오늘 13,428원 / 평소 16,601원 (-3,173원)
- 5.밤/야식(21-24): 오늘 1,250원 / 평소 10,228원 (-8,978원)"""


In [15]:
assert dummy_user_index_md.startswith("# 멤버 1번 일일 소비 분석 종합 리포트")
assert "## [1] 클리핑 데이터 → 안정적 지표 분석" in dummy_user_index_md
assert "## [2] 원본 데이터 → 행동 및 이상 탐지" in dummy_user_index_md
assert "## [3] 전날(2024-03-31) 대비 소비 비교 분석" in dummy_user_index_md
assert "## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)" in dummy_user_index_md
assert "SKT통신비" in dummy_user_index_md


In [16]:
FindingConfidence = Literal["low", "medium", "high"]


class EvidenceItem(BaseModel):
    """마크다운 보고서에서 참조한 근거 문장을 표현한다."""

    source_section: str = Field(description="근거가 나온 보고서 섹션 제목 또는 번호")
    supporting_text: str = Field(description="근거가 된 보고서 원문 일부")
    reason: str = Field(description="이 근거를 선택한 이유")


class SpendingFinding(BaseModel):
    """소비 분석의 단일 탐지 결과를 표현한다."""

    subcategory: str = Field(description="세부 분석 분류")
    title: str = Field(description="한 줄 요약")
    detail: str = Field(description="근거를 포함한 구체적인 설명")
    confidence: FindingConfidence = Field(description="판단 신뢰도")
    evidences: list[EvidenceItem] = Field(description="판단에 사용한 보고서 근거 목록")


class PatternAnalysisResult(BaseModel):
    """소비 패턴 탐지 결과를 구조화한다."""

    repeated_consumption: list[SpendingFinding] = Field(default_factory=list)
    overspending_windows: list[SpendingFinding] = Field(default_factory=list)
    impulse_patterns: list[SpendingFinding] = Field(default_factory=list)
    contextual_patterns: list[SpendingFinding] = Field(default_factory=list)


class ProblemAnalysisResult(BaseModel):
    """문제 소비 식별 결과를 구조화한다."""

    money_leaks: list[SpendingFinding] = Field(default_factory=list)
    saving_blockers: list[SpendingFinding] = Field(default_factory=list)
    fixed_cost_issues: list[SpendingFinding] = Field(default_factory=list)
    variable_cost_issues: list[SpendingFinding] = Field(default_factory=list)
    short_term_problem_spending: list[SpendingFinding] = Field(default_factory=list)
    long_term_problem_spending: list[SpendingFinding] = Field(default_factory=list)


class CauseAnalysisResult(BaseModel):
    """소비 원인 해석 결과를 구조화한다."""

    habitual_causes: list[SpendingFinding] = Field(default_factory=list)
    reward_causes: list[SpendingFinding] = Field(default_factory=list)
    stress_causes: list[SpendingFinding] = Field(default_factory=list)
    convenience_causes: list[SpendingFinding] = Field(default_factory=list)
    small_accumulation_causes: list[SpendingFinding] = Field(default_factory=list)


class ActionMission(BaseModel):
    """행동 개선 포인트와 실행 미션을 표현한다."""

    action_type: str = Field(description="행동 포인트 유형")
    title: str = Field(description="실행 항목 제목")
    detail: str = Field(description="실행 방법 설명")
    target_section: str = Field(description="직접 연결되는 보고서 섹션 또는 근거 영역")
    expected_effect: str = Field(description="기대 효과")
    urgency: Literal["immediate", "this_week", "this_month"] = Field(
        description="실행 우선순위 시점"
    )


class GroupCompetitionMetric(BaseModel):
    """그룹 경쟁에 반영할 행동 지표를 표현한다."""

    metric_name: str = Field(description="행동 지표 이름")
    definition: str = Field(description="행동 지표 계산 규칙")
    target_value: str = Field(description="권장 목표값")
    reason: str = Field(description="이 지표를 추천하는 이유")


class ActionAnalysisResult(BaseModel):
    """행동 개선 포인트 도출 결과를 구조화한다."""

    immediate_cuts: list[ActionMission] = Field(default_factory=list)
    substitution_opportunities: list[ActionMission] = Field(default_factory=list)
    budget_control_areas: list[ActionMission] = Field(default_factory=list)
    next_week_missions: list[ActionMission] = Field(default_factory=list)
    group_competition_metrics: list[GroupCompetitionMetric] = Field(default_factory=list)


def normalize_index_report(index_report_md: str) -> str:
    """마크다운 소비 보고서를 프롬프트 입력용 문자열로 정리한다."""
    return index_report_md.strip()


def make_analysis_input(index_report_md: str) -> dict[str, str]:
    """소비 분석 체인에 넣을 입력 페이로드를 생성한다."""
    return {"report_text": normalize_index_report(index_report_md)}


def build_pattern_prompt() -> ChatPromptTemplate:
    """마크다운 소비 보고서 기반 소비 패턴 탐지용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 카드 소비 보고서를 해석하는 금융 코치다. 제공된 보고서 내용만 사용해 판단하고, "
                "근거가 부족하면 confidence를 낮게 설정하라. 모든 응답은 한국어로 작성하고, "
                "evidences.source_section에는 보고서 섹션 제목 또는 [1]~[4] 번호를 적고, "
                "evidences.supporting_text에는 보고서 원문 일부를 짧게 인용하라.",
            ),
            (
                "human",
                "아래 사용자 소비 보고서를 바탕으로 소비 패턴을 탐지하라.\n"
                "반드시 반복 소비, 과소비 구간, 충동소비 의심 패턴, 시간대/상황별 소비 패턴을 각각 채워라.\n\n"
                "소비 보고서:\n{report_text}",
            ),
        ]
    )


def build_problem_prompt() -> ChatPromptTemplate:
    """마크다운 소비 보고서 기반 문제 소비 식별용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 사용자의 절약 실패 원인을 분리해서 설명하는 소비 분석가다. "
                "제공된 보고서 내용만 사용하고 추측성 서술은 최소화하라. "
                "고정비와 변동비 문제를 분리하고, 단기 문제와 장기 문제를 구분하라.",
            ),
            (
                "human",
                "아래 사용자 소비 보고서를 바탕으로 문제 소비를 식별하라.\n"
                "새는 돈 포인트, 절약 방해 요소, 고정비 문제, 변동비 문제, 단기 문제 소비, 장기 문제 소비를 채워라.\n\n"
                "소비 보고서:\n{report_text}",
            ),
        ]
    )


def build_cause_prompt() -> ChatPromptTemplate:
    """마크다운 소비 보고서 기반 소비 원인 해석용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 이미 식별된 패턴과 문제 소비를 바탕으로 행동 원인을 해석하는 분석가다. "
                "습관성, 보상성, 스트레스성, 편의성 기반, 소액 누적형 소비를 구분해 설명하라. "
                "증거가 약하면 낮은 confidence를 사용하라.",
            ),
            (
                "human",
                "아래 소비 보고서와 선행 분석 결과를 바탕으로 소비 원인을 해석하라.\n\n"
                "소비 보고서:\n{report_text}\n\n"
                "패턴 탐지 결과:\n{pattern_text}\n\n"
                "문제 소비 결과:\n{problem_text}",
            ),
        ]
    )


def build_action_prompt() -> ChatPromptTemplate:
    """마크다운 소비 보고서 기반 행동 개선 포인트 도출용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 소비 코칭 액션 플랜을 만드는 코치다. 즉시 줄일 수 있는 소비, 대체 가능한 소비, "
                "예산 통제가 필요한 영역, 다음 주 행동 미션, 그룹 경쟁 지표를 구체적으로 제안하라. "
                "행동 항목은 짧고 실행 가능해야 하며 target_section에는 보고서 섹션이나 근거 영역을 적어라.",
            ),
            (
                "human",
                "아래 소비 보고서와 선행 분석을 바탕으로 행동 개선 포인트를 도출하라.\n\n"
                "소비 보고서:\n{report_text}\n\n"
                "패턴 탐지 결과:\n{pattern_text}\n\n"
                "문제 소비 결과:\n{problem_text}\n\n"
                "소비 원인 결과:\n{cause_text}",
            ),
        ]
    )


In [17]:
analysis_input = make_analysis_input(dummy_user_index_md)

pattern_prompt = build_pattern_prompt()
problem_prompt = build_problem_prompt()
cause_prompt = build_cause_prompt()
action_prompt = build_action_prompt()

assert analysis_input["report_text"].startswith("# 멤버 1번 일일 소비 분석 종합 리포트")
assert set(pattern_prompt.input_variables) == {"report_text"}
assert set(problem_prompt.input_variables) == {"report_text"}
assert set(cause_prompt.input_variables) == {"report_text", "pattern_text", "problem_text"}
assert set(action_prompt.input_variables) == {
    "report_text",
    "pattern_text",
    "problem_text",
    "cause_text",
}

rendered_pattern = pattern_prompt.invoke(analysis_input)
assert "SKT통신비" in rendered_pattern.messages[-1].content
assert "반드시 반복 소비" in rendered_pattern.messages[-1].content

rendered_cause = cause_prompt.invoke(
    {
        "report_text": analysis_input["report_text"],
        "pattern_text": "sample pattern",
        "problem_text": "sample problem",
    }
)
assert "sample pattern" in rendered_cause.messages[-1].content
assert "sample problem" in rendered_cause.messages[-1].content


In [18]:
def prepare_cause_payload(payload: dict[str, object]) -> dict[str, str]:
    """원인 해석 체인에 필요한 입력 페이로드를 생성한다."""
    pattern_result = payload["pattern_result"]
    problem_result = payload["problem_result"]
    assert isinstance(pattern_result, PatternAnalysisResult)
    assert isinstance(problem_result, ProblemAnalysisResult)
    return {
        "report_text": str(payload["report_text"]),
        "pattern_text": pattern_result.model_dump_json(indent=2),
        "problem_text": problem_result.model_dump_json(indent=2),
    }


def prepare_action_payload(payload: dict[str, object]) -> dict[str, str]:
    """행동 개선 체인에 필요한 입력 페이로드를 생성한다."""
    pattern_result = payload["pattern_result"]
    problem_result = payload["problem_result"]
    cause_result = payload["cause_result"]
    assert isinstance(pattern_result, PatternAnalysisResult)
    assert isinstance(problem_result, ProblemAnalysisResult)
    assert isinstance(cause_result, CauseAnalysisResult)
    return {
        "report_text": str(payload["report_text"]),
        "pattern_text": pattern_result.model_dump_json(indent=2),
        "problem_text": problem_result.model_dump_json(indent=2),
        "cause_text": cause_result.model_dump_json(indent=2),
    }


def create_notebook_llm(model_name: str = "gpt-4o-mini") -> ChatOpenAI:
    """노트북 실행용 ChatOpenAI 모델을 생성한다."""
    return ChatOpenAI(model=model_name, temperature=0)


def build_pattern_chain(llm: ChatOpenAI):
    """소비 패턴 탐지 체인을 생성한다."""
    return build_pattern_prompt() | llm.with_structured_output(PatternAnalysisResult)


def build_problem_chain(llm: ChatOpenAI):
    """문제 소비 식별 체인을 생성한다."""
    return build_problem_prompt() | llm.with_structured_output(ProblemAnalysisResult)


def build_cause_chain(llm: ChatOpenAI):
    """소비 원인 해석 체인을 생성한다."""
    return build_cause_prompt() | llm.with_structured_output(CauseAnalysisResult)


def build_action_chain(llm: ChatOpenAI):
    """행동 개선 포인트 도출 체인을 생성한다."""
    return build_action_prompt() | llm.with_structured_output(ActionAnalysisResult)


def build_spending_analysis_chain(llm: ChatOpenAI):
    """소비 패턴, 문제 소비, 원인, 행동 포인트를 연결한 다단계 체인을 생성한다."""
    diagnosis_chain = RunnableParallel(
        report_text=RunnableLambda(lambda payload: str(payload["report_text"])),
        pattern_result=build_pattern_chain(llm),
        problem_result=build_problem_chain(llm),
    )
    return (
        diagnosis_chain
        | RunnablePassthrough.assign(
            cause_result=RunnableLambda(prepare_cause_payload) | build_cause_chain(llm)
        )
        | RunnablePassthrough.assign(
            action_result=RunnableLambda(prepare_action_payload) | build_action_chain(llm)
        )
    )


In [19]:
pattern_prompt_preview = build_pattern_prompt().invoke(analysis_input).messages[-1].content
print(pattern_prompt_preview[:1200])


아래 사용자 소비 보고서를 바탕으로 소비 패턴을 탐지하라.
반드시 반복 소비, 과소비 구간, 충동소비 의심 패턴, 시간대/상황별 소비 패턴을 각각 채워라.

소비 보고서:
# 멤버 1번 일일 소비 분석 종합 리포트

- 분석 기준일: 2024-04-01
- 비교 기준일: 2024-03-31

## [1] 클리핑 데이터 → 안정적 지표 분석
- 과거 일평균(안정): 51,014원
- 오늘 총 지출액: 133,044원
- 평소 대비 지출 증가율: +160.80%

### 카테고리 비중 변화
#### 비중 증가 상위
- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)
- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)
- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)

#### 비중 감소 상위
- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)
- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)

## [2] 원본 데이터 → 행동 및 이상 탐지
- 과거 원본 일평균(전체): 104,424원
- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)

### 특이 지출 내역
- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활

## [3] 전날(2024-03-31) 대비 소비 비교 분석
- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)
- 결제 건수 비교: 6건 -> 9건 (+3건)
- 주 소비 카테고리 변화: 식비 -> 생활

## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
- 오늘의 소비 피크 타임: 2.오전(06-11)

### 시간대별 세부 비교
- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)
- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)
- 4.저녁(1

In [20]:
openai_api_key = os.getenv("OPENAI_API_KEY", "").strip()

if openai_api_key:
    spending_analysis_chain = build_spending_analysis_chain(create_notebook_llm())
    analysis_result = spending_analysis_chain.invoke(analysis_input)
    print(analysis_result)
else:
    analysis_result = None
    print(
        "OPENAI_API_KEY가 없어 체인 실행은 건너뜁니다. 프롬프트와 체인 정의 셀까지만 검증했습니다."
    )


{'report_text': '# 멤버 1번 일일 소비 분석 종합 리포트\n\n- 분석 기준일: 2024-04-01\n- 비교 기준일: 2024-03-31\n\n## [1] 클리핑 데이터 → 안정적 지표 분석\n- 과거 일평균(안정): 51,014원\n- 오늘 총 지출액: 133,044원\n- 평소 대비 지출 증가율: +160.80%\n\n### 카테고리 비중 변화\n#### 비중 증가 상위\n- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)\n- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)\n- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)\n\n#### 비중 감소 상위\n- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)\n- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)\n\n## [2] 원본 데이터 → 행동 및 이상 탐지\n- 과거 원본 일평균(전체): 104,424원\n- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)\n\n### 특이 지출 내역\n- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활\n\n## [3] 전날(2024-03-31) 대비 소비 비교 분석\n- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)\n- 결제 건수 비교: 6건 -> 9건 (+3건)\n- 주 소비 카테고리 변화: 식비 -> 생활\n\n## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)\n- 오늘의 소비 피크 타임: 2.오전(06-11)\n\n### 시간대별 세부 비교\n- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)\n- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)\n- 4.저녁(17-21): 오늘 13,428원 / 평소 16,601원 (-3,173원)\

In [21]:
def print_pretty_report(result: object | None) -> None:
    """체인 결과를 입력 보고서와 함께 보기 쉽게 출력한다."""
    if not result:
        print("분석 결과가 없습니다. 체인 실행 여부를 먼저 확인하세요.")
        return

    if isinstance(result, dict) and "report_text" in result:
        print("[1. 입력된 소비 보고서]")
        print("=" * 60)
        print(result["report_text"])
        print("=" * 60 + "\n")

    categories = [
        ("pattern_result", "[2. 소비 패턴 분석]"),
        ("problem_result", "[3. 문제 소비 식별]"),
        ("cause_result", "[4. 소비 원인 해석]"),
        ("action_result", "[5. 행동 개선 제안]"),
    ]

    if isinstance(result, dict):
        for key, title in categories:
            if key not in result:
                continue
            print(title)
            print("-" * 60)
            data = result[key]
            if hasattr(data, "model_dump_json"):
                print(data.model_dump_json(indent=2))
            else:
                print(data)
            print("-" * 60 + "\n")


print_pretty_report(analysis_result)


[1. 입력된 소비 보고서]
# 멤버 1번 일일 소비 분석 종합 리포트

- 분석 기준일: 2024-04-01
- 비교 기준일: 2024-03-31

## [1] 클리핑 데이터 → 안정적 지표 분석
- 과거 일평균(안정): 51,014원
- 오늘 총 지출액: 133,044원
- 평소 대비 지출 증가율: +160.80%

### 카테고리 비중 변화
#### 비중 증가 상위
- 생활: 평소 1.5% -> 오늘 72.8% (+71.4pt)
- 교통: 평소 4.7% -> 오늘 9.7% (+4.9pt)
- 의료: 평소 8.1% -> 오늘 10.1% (+2.0pt)

#### 비중 감소 상위
- 식비: 평소 68.4% -> 오늘 7.4% (-61.0pt)
- 쇼핑: 평소 17.3% -> 오늘 0.0% (-17.3pt)

## [2] 원본 데이터 → 행동 및 이상 탐지
- 과거 원본 일평균(전체): 104,424원
- 소비 규모 판단: ✅ 소비 규모가 평소 범위를 유지하고 있습니다. (평소 대비 1.3배)

### 특이 지출 내역
- 2024-04-01 10:00 | SKT통신비 | 65,000원 | 생활

## [3] 전날(2024-03-31) 대비 소비 비교 분석
- 지출액 비교: 2024-03-31 48,531원 -> 2024-04-01 133,044원 (+84,513원, +174.1%)
- 결제 건수 비교: 6건 -> 9건 (+3건)
- 주 소비 카테고리 변화: 식비 -> 생활

## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
- 오늘의 소비 피크 타임: 2.오전(06-11)

### 시간대별 세부 비교
- 2.오전(06-11): 오늘 112,895원 / 평소 9,496원 (+103,399원)
- 3.점심/오후(11-17): 오늘 5,471원 / 평소 68,099원 (-62,628원)
- 4.저녁(17-21): 오늘 13,428원 / 평소 16,601원 (-3,173원)
- 5.밤/야식(21-24): 오늘 1,250원 / 평소 10,228원